# Benchmark: Data Lakehouse (Iceberg/Parquet) vs Raw CSV

Notebook ini berfungsi untuk membuktikan dua keunggulan utama arsitektur Data Lakehouse:
1. **Storage Efficiency:** Membandingkan ukuran data mentah (CSV) vs data Lakehouse (Parquet kompresi Snappy) secara spesifik untuk setiap tabel TPC-H, beserta total penghematannya.
2. **Query Performance:** Mengadu kecepatan eksekusi query agregasi analitik (GROUP BY) antara membaca file CSV secara langsung vs membaca dari tabel Iceberg.

In [3]:
import time
import sys
import os
from minio import Minio
from trino.dbapi import connect


sys.path.append(os.path.abspath('../code'))
from upload_csv_to_lakehouse import format_file_size

# MinIO Config
MINIO_ENDPOINT = "localhost:9005"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"

# Trino Config
TRINO_HOST = "localhost"
TRINO_PORT = 8080
TRINO_USER = "trino"

## 1. Storage Size Comparison (Per Table)
Menggunakan MinIO API untuk menelusuri folder data mentah (`csv/`) dan folder Lakehouse (`iceberg/tpch/`), lalu membandingkan ukuran penyimpanannya tabel demi tabel.

In [4]:
client = Minio(endpoint=MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)

def get_total_size(bucket_name, prefix=""):
    try:
        objects = client.list_objects(bucket_name, prefix=prefix, recursive=True)
        return sum(obj.size for obj in objects)
    except Exception:
        return 0

tables = ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]

total_csv_all = 0
total_iceberg_all = 0

print("=" * 70)
print(f"{'NAMA TABEL':<15} | {'RAW CSV SIZE':<15} | {'LAKEHOUSE SIZE':<15} | {'PENGHEMATAN':>15}")
print("-" * 70)

for table in tables:
    # ukuran CSV di bucket 'lakehouse'
    csv_size = get_total_size("lakehouse", f"csv/{table}/")
    
    # ukuran Iceberg (Parquet + Metadata) di bucket 'iceberg'
    iceberg_size = get_total_size("iceberg", f"tpch/{table}/")
    
    total_csv_all += csv_size
    total_iceberg_all += iceberg_size
    
    # prevent error pembagian dengan nol
    if csv_size > 0:
        savings_pct = (1 - (iceberg_size / csv_size)) * 100
    else:
        savings_pct = 0.0
        
    print(f"{table:<15} | {format_file_size(csv_size):<15} | {format_file_size(iceberg_size):<15} | {savings_pct:>14.2f}%")

print("-" * 70)

if total_csv_all > 0:
    total_savings_pct = (1 - (total_iceberg_all / total_csv_all)) * 100
else:
    total_savings_pct = 0.0

print(f"{'GRAND TOTAL':<15} | {format_file_size(total_csv_all):<15} | {format_file_size(total_iceberg_all):<15} | {total_savings_pct:>14.2f}%")
print("=" * 70)

NAMA TABEL      | RAW CSV SIZE    | LAKEHOUSE SIZE  |     PENGHEMATAN
----------------------------------------------------------------------
customer        |  23.38 MB       |   8.32 MB       |          64.39%
lineitem        | 725.75 MB       | 136.09 MB       |          81.25%
nation          |   2.19 KB       |  23.73 KB       |        -985.70%
orders          | 164.47 MB       |  37.11 MB       |          77.44%
part            |  23.04 MB       |   4.54 MB       |          80.28%
partsupp        | 114.04 MB       |  28.74 MB       |          74.80%
region          |   0.38 KB       |  21.77 KB       |       -5601.02%
supplier        |   1.35 MB       | 731.93 KB       |          47.21%
----------------------------------------------------------------------
GRAND TOTAL     |   1.03 GB       | 215.57 MB       |          79.51%


## 2. Query Performance Benchmark (Trino)
Menjalankan *query* TPC-H Query 1 (Pricing Summary Report) versi sederhana yang menggunakan operasi `GROUP BY` dan `SUM()` pada tabel `lineitem`.

In [5]:
import time
from trino.dbapi import connect

def run_benchmark_query(catalog, schema, query_type):
    # Connect ke Trino
    conn = connect(host=TRINO_HOST, port=TRINO_PORT, user=TRINO_USER, catalog=catalog, schema=schema)
    cursor = conn.cursor()
    
    # 1. NORMAL QUERY (Single Table Aggregation)
    if query_type == "normal":
        query = """
            SELECT 
                l_returnflag, 
                l_linestatus, 
                SUM(CAST(l_quantity AS DOUBLE)) as sum_qty,
                COUNT(*) as count_order
            FROM lineitem
            GROUP BY l_returnflag, l_linestatus
            ORDER BY l_returnflag, l_linestatus
        """
        
    # 2. HEAVY QUERY (3-Table JOIN + Aggregation)
    elif query_type == "heavy":
        query = """
            SELECT 
                c.c_mktsegment,
                SUM(CAST(l.l_extendedprice AS DOUBLE) * (1 - CAST(l.l_discount AS DOUBLE))) as revenue
            FROM customer c
            JOIN orders o ON CAST(c.c_custkey AS BIGINT) = CAST(o.o_custkey AS BIGINT)
            JOIN lineitem l ON CAST(o.o_orderkey AS BIGINT) = CAST(l.l_orderkey AS BIGINT)
            WHERE CAST(o.o_orderdate AS DATE) >= DATE '1994-01-01' 
              AND CAST(o.o_orderdate AS DATE) < DATE '1995-01-01'
              AND CAST(l.l_shipdate AS DATE) >= DATE '1994-01-01'
              AND CAST(l.l_shipdate AS DATE) < DATE '1995-01-01'
            GROUP BY c.c_mktsegment
            ORDER BY revenue DESC
        """
    
    start_time = time.time()
    cursor.execute(query)
    results = cursor.fetchall() # Fetch agar eksekusi selesai total
    end_time = time.time()
    
    cursor.close()
    conn.close()
    
    return end_time - start_time


print("=" * 60)
print("1. NORMAL QUERY BENCHMARK (Single Table)")
print("=" * 60)

try:
    time_csv_normal = run_benchmark_query("hive", "tpch_external", "normal")
    print(f"-> Waktu Eksekusi CSV     : {time_csv_normal:.2f} detik")
except Exception as e:
    print(f"-> Gagal eksekusi CSV: {e}")
    time_csv_normal = None

try:
    time_iceberg_normal = run_benchmark_query("iceberg", "tpch", "normal")
    print(f"-> Waktu Eksekusi Iceberg : {time_iceberg_normal:.2f} detik")
except Exception as e:
    print(f"-> Gagal eksekusi Iceberg: {e}")
    time_iceberg_normal = None

if time_csv_normal and time_iceberg_normal:
    print("-" * 60)
    print(f"KESIMPULAN NORMAL: Lakehouse {time_csv_normal / time_iceberg_normal:.1f}x lebih cepat\n")



print("=" * 60)
print("2. HEAVY QUERY BENCHMARK (3-Table JOIN)")
print("=" * 60)

try:
    time_csv_heavy = run_benchmark_query("hive", "tpch_external", "heavy")
    print(f"-> Waktu Eksekusi CSV     : {time_csv_heavy:.2f} detik")
except Exception as e:
    print(f"-> Gagal eksekusi CSV: {e}")
    time_csv_heavy = None

try:
    time_iceberg_heavy = run_benchmark_query("iceberg", "tpch", "heavy")
    print(f"-> Waktu Eksekusi Iceberg : {time_iceberg_heavy:.2f} detik")
except Exception as e:
    print(f"-> Gagal eksekusi Iceberg: {e}")
    time_iceberg_heavy = None

if time_csv_heavy and time_iceberg_heavy:
    print("-" * 60)
    print(f"KESIMPULAN HEAVY : Lakehouse {time_csv_heavy / time_iceberg_heavy:.1f}x lebih cepat")
print("=" * 60)

1. NORMAL QUERY BENCHMARK (Single Table)
-> Waktu Eksekusi CSV     : 1.37 detik
-> Waktu Eksekusi Iceberg : 1.36 detik
------------------------------------------------------------
KESIMPULAN NORMAL: Lakehouse 1.0x lebih cepat

2. HEAVY QUERY BENCHMARK (3-Table JOIN)
-> Waktu Eksekusi CSV     : 3.18 detik
-> Waktu Eksekusi Iceberg : 1.93 detik
------------------------------------------------------------
KESIMPULAN HEAVY : Lakehouse 1.6x lebih cepat
